# 7B · Deciding Under Uncertainty — Trees, Scenarios, Break-Evens
### Financial Analytics — Module 7

7A optimised with *known* inputs. Real decisions rarely get that luxury: outcomes are uncertain, and the tool changes from a solver to a **decision tree** — the disciplined way to choose when the future forks.

The scenario: MoneyMart's board is weighing a new small-format store concept.

- **Launch big now:** roll out 40 stores. Cost ₹120 cr. If the concept works (management's estimate: 55%), payoff ₹300 cr NPV; if it flops, salvage ₹40 cr.
- **Pilot first:** 4 stores for ₹15 cr, one year. The pilot reveals whether the concept works, *then* the rollout decision is made with that knowledge. (Assume, for now, a perfectly informative pilot — we'll price that assumption.)
- **Don't launch:** ₹0.

Which branch should the board take — and what is the pilot actually worth?

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

p_success = 0.55
COST_BIG, PAYOFF_WIN, SALVAGE = 120, 300, 40
COST_PILOT = 15

---
## 1. Expected Monetary Value: the spine of the tree

For each choice, weight every outcome by its probability and sum. That's **EMV** — the long-run average value of taking this branch many times.

In [ ]:
# Branch 1: launch big now
emv_launch = p_success*(PAYOFF_WIN - COST_BIG) + (1-p_success)*(SALVAGE - COST_BIG)

# Branch 2: pilot, then decide WITH KNOWLEDGE
#   If pilot says works (prob p): roll out -> payoff 300 - 120. If says flop: walk away -> 0.
#   Either way the pilot itself cost 15.
emv_pilot = p_success*(PAYOFF_WIN - COST_BIG) + (1-p_success)*0 - COST_PILOT

emv_no = 0.0

for name, v in [("Launch big now", emv_launch), ("Pilot first", emv_pilot), ("Don't launch", emv_no)]:
    print(f"{name:<16} EMV = Rs {v:+7.1f} cr")

**Read the ranking.** The pilot wins — not because it's cautious, but because of what it *buys*: the right to walk away from the flop branch. Launching big carries the full downside `(40 − 120) = −80` weighted at 45%; the pilot converts that branch into a clean 0 (minus the pilot's price).

That right-to-walk-away has a formal name and a price:

## 2. The value of information

In [ ]:
# What would a PERFECT crystal ball be worth, before knowing its price?
# With perfect info: launch only in the success world.
emv_perfect_free = p_success*(PAYOFF_WIN - COST_BIG) + (1-p_success)*0
evpi = emv_perfect_free - max(emv_launch, emv_no)     # vs best choice WITHOUT info

print(f"Best without information : Rs {max(emv_launch, emv_no):.1f} cr  (launch)")
print(f"With free perfect info   : Rs {emv_perfect_free:.1f} cr")
print(f"Value of perfect info    : Rs {evpi:.1f} cr   <- the MOST any study/pilot can be worth")
print(f"The pilot costs Rs {COST_PILOT} cr  ->  a bargain, IF it's truly informative.")

**EVPI — the Expected Value of Perfect Information — is a ceiling.** No consultant's study, no pilot, no market research can rationally cost more than it. Computing EVPI *before* commissioning any research is one of the most quietly powerful habits in decision analysis: it prices curiosity.

*(Honesty note: our pilot was assumed perfectly informative. Real pilots misfire — small samples, unrepresentative sites — so a real pilot is worth less than EVPI. The machinery for imperfect information exists (Bayes' rule); the instinct — "information has a computable ceiling" — is today's takeaway.)*

---
## 3. The break-even probability: attack your own assumption

Everything above leaned on one soft number: management's 55%. The classic counter is not to argue about 55 — it's to ask: **at what probability does the decision flip?**

In [ ]:
ps = np.linspace(0, 1, 101)
launch_line = ps*(PAYOFF_WIN-COST_BIG) + (1-ps)*(SALVAGE-COST_BIG)
pilot_line  = ps*(PAYOFF_WIN-COST_BIG) - COST_PILOT

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(ps, launch_line, label="Launch big now", color="#DC2626", lw=2)
ax.plot(ps, pilot_line,  label="Pilot first",   color="#2563EB", lw=2)
ax.axhline(0, color="black", lw=0.8, label="Don't launch")
ax.axvline(0.55, color="grey", ls=":", lw=1)
ax.text(0.555, -90, "management's 55%", fontsize=8, color="grey")

# Where do the lines cross?
be_pilot_vs_no  = COST_PILOT/(PAYOFF_WIN-COST_BIG)
be_launch_vs_pilot = None   # launch beats pilot when its line is higher: solve directly
# launch - pilot = (1-p)(SALVAGE-COST_BIG) + COST_PILOT = 0  ->  p = 1 + COST_PILOT/(SALVAGE-COST_BIG)
be = 1 + COST_PILOT/(SALVAGE-COST_BIG)
print(f"Pilot beats 'don't launch' above p = {be_pilot_vs_no:.0%}")
print(f"Launch-now overtakes pilot only above p = {be:.0%}")
ax.set_xlabel("probability the concept works"); ax.set_ylabel("EMV (Rs cr)")
ax.set_title("The decision as a function of the belief - where does it FLIP?", loc="left", fontweight="bold")
ax.legend(); plt.tight_layout(); plt.show()

**This chart is the professional deliverable** — better than any single recommendation. It says: *pilot dominates across essentially every plausible belief; launch-now only wins if you're nearly certain (~81%+); below ~8% do nothing.* Now the board argument about "is it 50% or 60%?" — the argument that consumes most meetings — is revealed as **irrelevant to the decision**. Both beliefs land on the same branch.

The move has a name worth using in interviews: convert *"what is the probability?"* (unanswerable) into *"is the probability above the break-even?"* (usually answerable with confidence).

### ✏️ Exercise 1
Salvage improves: a franchisee will buy failed stores, raising salvage from ₹40 to ₹90 cr. Recompute all EMVs and the launch-vs-pilot break-even. Why does *better downside protection* specifically erode the **pilot's** advantage?

### ✏️ Exercise 2
The board is risk-averse: they weigh losses 2× gains (multiply every negative outcome by 2 before averaging). Re-rank the three branches. Which choice is most robust to this change — and why is "robust to the utility function" itself a selling point when you present?

In [ ]:
# your code here


---
## 4. Scenario weighting: prescriptive with a forecast attached

Trees fork on events; **scenarios** fork on *worlds*. The treasury allocation from 7A assumed one rate world — but the desk's economist offers three:

| Scenario | Prob | Overnight | T-bills | FD | Corp |
|---|---|---|---|---|---|
| Rates fall | 30% | 5.4% | 6.0% | 6.8% | 8.0% |
| Base | 50% | 6.2% | 6.8% | 7.1% | 8.4% |
| Rates rise | 20% | 7.2% | 7.8% | 7.3% | 8.6% |

In [ ]:
from scipy.optimize import linprog
scen = pd.DataFrame({
    "prob": [0.30, 0.50, 0.20],
    "Overnight": [0.054, 0.062, 0.072],
    "T-bills":   [0.060, 0.068, 0.078],
    "Bank FD":   [0.068, 0.071, 0.073],
    "Corp":      [0.080, 0.084, 0.086],
}, index=["fall","base","rise"])

# Probability-weighted expected yields - then optimise ONCE against them
exp_yields = (scen[["Overnight","T-bills","Bank FD","Corp"]].T @ scen["prob"]).values

r = linprog(-exp_yields, A_ub=[[-1,-1,0,0],[0,0,0,1],[0,0,1,0]], b_ub=[-30,20,35],
            A_eq=[[1,1,1,1]], b_eq=[100], bounds=[(0,None)]*4, method="highs")
alloc = pd.Series(r.x, index=["Overnight","T-bills","Bank FD","Corp"])
print("Allocation vs expected yields:\n", alloc.round(1).to_string())

# The stress question: how does THIS allocation fare in EACH world?
for s in scen.index:
    y = (alloc.values * scen.loc[s, ["Overnight","T-bills","Bank FD","Corp"]].values).sum()
    print(f"  in '{s}' world: Rs {y:.2f} cr/yr")

Two habits in one cell: **optimise against the probability-weighted world**, then **stress the answer in each scenario separately** — because a decision that's optimal on average but catastrophic in one plausible world is often the wrong decision. (When "catastrophic in one world" is the dominant concern, the framework changes name — robust optimisation, maximin — and the instinct you just practised is its doorway.)

### ✏️ Exercise 3
Re-solve the allocation *separately inside each scenario*. How different are the three "optimal" answers? What does that spread tell you about how confidently anyone should present the single weighted answer?

---
## The ladder, complete

Describe → diagnose → predict → **prescribe**. Notice what this module consumed: clean data (M1/3), decomposed drivers (M5), forecasts with stated uncertainty (M6) — all feeding objectives, probabilities and constraints. Prescriptive analytics is not a technique; it's the ladder, load-bearing, with a decision on top.

*(There is a fifth rung the industry now talks about — **agentic**: systems that decide AND act, loops closed, humans supervising. Everything you just learned is the safety case for it: an agent is an optimiser with hands, and every warning label in this module applies at machine speed. Awareness is all this course owes you on it.)*

---
*AI disclosure: ______*

In [ ]:
# workspace
